# Ethylene case study

This notebook demonstrates environmental and economic optimization of three
ethylene routes with optimex. All results remain provisional while proxy assumptions
are present.


## 1. Data and scenario

The study covers Europe from 2025 to 2050 and uses the
`REMIND-EU_SSP2-NDC` premise databases. Annual ethylene demand is constant at
`1e9 kg/a`. The routes are steam cracking, methanol-to-olefins supplied by DAC,
PEM electrolysis and CO2 hydrogenation, and an aggregated eCO2R process that
includes product separation.

Case-study-specific background activities are stored in four small interface
databases. They link to the corresponding premise activities for each support
year while leaving the premise databases unchanged.

The use phase, ethylene product end-of-life and plant end-of-life are excluded.
The retained `eol=yes` eCO2R contribution represents oxidation of process
by-products, not ethylene product end-of-life.

In [ ]:
from datetime import datetime
from pathlib import Path

import bw2data as bd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyomo.environ as pyo
from bw_temporalis import TemporalDistribution
from IPython.display import display

from optimex.utils import (
    infer_construction_td_from_limits,
    infer_operation_td_from_limits,
)

PROJECT = "optimex_remind"
BIOSPHERE_DB = "ecoinvent-3.12-biosphere"
FOREGROUND_DB = "ethylene_case_study_foreground"
PREMISE_DBS = {
    2020: "ei312_REMIND-EU_SSP2_NDC_2020",
    2030: "ei312_REMIND-EU_SSP2_NDC_2030",
    2040: "ei312_REMIND-EU_SSP2_NDC_2040",
    2050: "ei312_REMIND-EU_SSP2_NDC_2050",
}
CASE_STUDY_BG_DBS = {
    year: f"ethylene_case_study_background_{year}"
    for year in PREMISE_DBS
}

bd.projects.set_current(PROJECT)
required_databases = {BIOSPHERE_DB, *PREMISE_DBS.values()}
missing_databases = required_databases - set(bd.databases)
if missing_databases:
    raise ValueError(f"Missing databases: {sorted(missing_databases)}")

## 2. Model assumptions

The table collects the assumptions that control plant lifetimes, installation
scaling, existing capacity and discounting. Installation-edge amounts are copied
unchanged from their source inventories. Process lifetimes only control Optimex
vintage availability and replacement; they are not used to rescale inventory
coefficients. Final runs require the corrected `var_installation` semantics to
pass a multi-year static-LCA equivalence test. The brownfield capacities must
also be revalidated against that corrected variable definition.


In [ ]:
assumptions = pd.DataFrame(
    [
        ("lifetime.steam_cracking", 25, "year", "RESEARCHED", "Tiggeloven (2026), Table C.1, conventional cracker", False),
        ("lifetime.dac", 20, "year", "RESEARCHED", "Deutz and Bardow (2021), https://doi.org/10.1038/s41560-020-00771-9", False),
        ("lifetime.pem", 25, "year", "PROXY", "Tiggeloven (2026), Table C.1, AEC plant; separate 9-year stack replacement not modeled", True),
        ("lifetime.co2_hydrogenation", 25, "year", "RESEARCHED", "Tiggeloven (2026), Table C.1, direct methanol synthesis from CO2", False),
        ("lifetime.mto", 25, "year", "RESEARCHED", "Tiggeloven (2026), Table C.1, methanol-to-olefins", False),
        ("lifetime.eco2r", 25, "year", "PROXY", "Tiggeloven (2026), Table C.1, CO2 electrolysis and air-separation-unit proxies", True),
        (
            "installation.steam_factory",
            1.1516356618335166e-10,
            "unit/kg ethylene",
            "INVENTORY",
            "ecoinvent coefficient retained outside the route-specific wrapper",
            False,
        ),
        (
            "installation.dac_system",
            1.25e-8,
            "unit/kg CO2",
            "INVENTORY",
            "disco2very 4 kt CO2/a DAC inventory coefficient copied unchanged",
            False,
        ),
        (
            "installation.pem_stack",
            1.34989e-6,
            "unit/kg H2",
            "INVENTORY",
            "methanol_and_iron inventory coefficient copied unchanged",
            False,
        ),
        (
            "installation.pem_bop",
            3.37373e-7,
            "unit/kg H2",
            "INVENTORY",
            "methanol_and_iron inventory coefficient copied unchanged",
            False,
        ),
        (
            "installation.methanol_factory",
            3.5842e-12,
            "unit/kg methanol",
            "INVENTORY",
            "disco2very inventory coefficient copied unchanged",
            False,
        ),
        (
            "installation.mto_factory",
            3.584e-12,
            "unit/kg ethylene",
            "INVENTORY",
            "disco2very coefficient retained outside the route-specific wrapper",
            False,
        ),
        (
            "installation.eco2r_factory",
            7.32e-7,
            "kg installation proxy/kg ethylene",
            "INVENTORY",
            "disco2very factory coefficient retained outside the aggregated wrapper",
            False,
        ),
        (
            "installation.eco2r_copper",
            7.4e-11,
            "kg/kg ethylene",
            "INVENTORY",
            "Internal input of the aggregated eCO2R installation wrapper",
            False,
        ),
        (
            "installation.eco2r_steel",
            3.017679e-6,
            "kg/kg ethylene",
            "INVENTORY",
            "Internal input of the aggregated eCO2R installation wrapper",
            False,
        ),
        ("brownfield.vintage_1", 2005, "year", "PROXY", "methanol_and_iron brownfield pattern", True),
        ("brownfield.vintage_2", 2015, "year", "PROXY", "methanol_and_iron brownfield pattern", True),
        ("brownfield.capacity_per_vintage", 0.5e9, "kg ethylene/a", "PROXY", "Two equal vintages", False),
        ("economics.discount_rate", 0.03, "real fraction/a", "PROXY", "basic_example_econ", True),
    ],
    columns=[
        "parameter", "value", "unit", "status", "basis", "replacement_needed"
    ],
).set_index("parameter")

PROCESS_NAMES = {
    "steam_cracking": "Steam cracking",
    "dac": "Direct air capture",
    "pem": "PEM electrolysis",
    "co2_hydrogenation": "CO2 hydrogenation to methanol",
    "mto": "Methanol-to-olefins",
    "eco2r": "eCO2R ethylene production",
}
LIFETIMES_YEARS = {
    process: int(assumptions.loc[f"lifetime.{process}", "value"])
    for process in PROCESS_NAMES
}
INSTALLATION_AMOUNTS = {
    key.removeprefix("installation."): float(row["value"])
    for key, row in assumptions.loc[
        assumptions.index.str.startswith("installation.")
    ].iterrows()
}

assumptions

## 3. Case-study background

The four case-study databases form the temporal background family seen by
optimex. Ordinary purchases are represented by transparent one-to-one interface
activities that link to the matching premise activity of the same year.

The databases also contain the seven-component steam-cracking feedstock mix,
route-specific installation wrappers and three DAC-specific activities reproduced
from disco2very. The DAC construction excludes plant end-of-life to retain the
cradle-to-gate system boundary. No activity is written to a premise database.

In [ ]:
PEI_NAME = "polyethyleneimine production"
PEI_PRODUCT = "polyethyleneimine"
ADSORBENT_NAME = "adsorbent, amine on alumina"
ADSORBENT_PRODUCT = "adsorbent, amine on alumina"
DAC_4KT_NAME = "direct air capture system construction, solid sorbent, 4 ktCO2/a"
DAC_4KT_PRODUCT = "direct air capture system, solid sorbent, 4 ktCO2/a"
FEEDSTOCK_MIX_NAME = "steam cracking feedstock mix"
STEAM_INSTALLATION_NAME = "steam cracker installation"
MTO_INSTALLATION_NAME = "methanol-to-olefins installation"
ECO2R_INSTALLATION_NAME = "eCO2R system installation"

STEAM_FEEDSTOCK_TOTAL = 1.21675954200327
STEAM_FEEDSTOCK_INPUTS = [
    ("market for butane", "butane", "RoW", "kilogram", 0.19711504876613617),
    ("market for ethane", "ethane", "RoW", "kilogram", 0.05475417897105217),
    ("market for naphtha", "naphtha", "RER", "kilogram", 0.7787261009216309),
    ("market for natural gas liquids", "natural gas liquids", "RoW", "kilogram", 0.02676870860159397),
    ("market for propane", "propane", "RoW", "kilogram", 0.09247373044490814),
    ("market for refinery gas", "refinery gas", "GLO", "kilogram", 0.018251389265060425),
    ("market for diesel", "diesel", "DEU", "kilogram", 0.04867038503289223),
]
assert np.isclose(
    sum(item[4] for item in STEAM_FEEDSTOCK_INPUTS),
    STEAM_FEEDSTOCK_TOTAL,
)

INTERFACE_SPECS = [
    ("cooling_minus_25", "cooling energy production, at -25 °C, propylene compression refrigeration system 1 MW", "cooling energy, at -25 °C", "GLO", "megajoule"),
    ("pem_bop", "electrolyzer production, 1MWe, PEM, Balance of Plant", "electrolyzer, 1MWe, PEM, Balance of Plant", "RER", "unit"),
    ("pem_stack", "electrolyzer production, 1MWe, PEM, Stack", "electrolyzer, 1MWe, PEM, Stack", "RER", "unit"),
    ("heat_pump", "heat production, at heat pump 30kW, allocation exergy", "heat, central or small-scale, other than natural gas", "Europe without Switzerland", "megajoule"),
    ("compressed_air", "market for compressed air, 700 kPa gauge", "compressed air, 700 kPa gauge", "RER", "cubic meter"),
    ("cooling", "market for cooling energy", "cooling energy", "GLO", "megajoule"),
    ("cooling_minus_100", "market for cooling energy, at -100 °C", "cooling energy, at -100 °C", "GLO", "megajoule"),
    ("cooling_minus_15", "market for cooling energy, at -15 °C", "cooling energy, at -15 °C", "GLO", "megajoule"),
    ("cooling_minus_45", "market for cooling energy, at -45 °C", "cooling energy, at -45 °C", "GLO", "megajoule"),
    ("cooling_minus_55", "market for cooling energy, at -55 °C", "cooling energy, at -55 °C", "GLO", "megajoule"),
    ("electricity_mv", "market for electricity, medium voltage", "electricity, medium voltage", "DE", "kilowatt hour"),
    ("hazardous_waste_incineration", "market for hazardous waste, for incineration", "hazardous waste, for incineration", "CH", "kilogram"),
    ("hazardous_waste_underground", "market for hazardous waste, for underground deposit", "hazardous waste, for underground deposit", "RER", "kilogram"),
    ("industrial_heat", "market for heat, district or industrial, natural gas", "heat, district or industrial, natural gas", "Europe without Switzerland", "megajoule"),
    ("inert_waste", "market for inert waste, for final disposal", "inert waste, for final disposal", "CH", "kilogram"),
    ("background_methanol", "market for methanol", "methanol", "RER w/o RU", "kilogram"),
    ("nitrogen_liquid", "market for nitrogen, liquid", "nitrogen, liquid", "RER", "kilogram"),
    ("sodium_hydroxide", "market for sodium hydroxide, without water, in 50% solution state", "sodium hydroxide, without water, in 50% solution state", "RER", "kilogram"),
    ("wastewater_average", "market for wastewater, average", "wastewater, average", "CH", "cubic meter"),
    ("wastewater_unpolluted", "market for wastewater, unpolluted", "wastewater, unpolluted", "RoW", "cubic meter"),
    ("water_deionised_ch", "market for water, deionised", "water, deionised", "CH", "kilogram"),
    ("water_deionised_europe", "market for water, deionised", "water, deionised", "Europe without Switzerland", "kilogram"),
    ("electricity_mv_deu", "market group for electricity, medium voltage", "electricity, medium voltage", "DEU", "kilowatt hour"),
    ("methanol_facility", "methanol production facility, construction", "methanol production facility, construction", "RER", "unit"),
    ("spent_adsorbent_treatment", "treatment of spent anion exchange resin from potable water production, municipal incineration", "spent anion exchange resin from potable water production", "GLO", "kilogram"),
]


def premise_node(database_name, *, name, product, location, unit):
    return bd.get_node(
        database=database_name,
        name=name,
        product=product,
        location=location,
        unit=unit,
    )


def create_case_activity(database, *, code, name, product, location, unit, comment):
    activity = database.new_node(
        code=code,
        name=name,
        product=product,
        location=location,
        unit=unit,
        type=bd.labels.process_node_default,
    )
    activity["reference product"] = product
    activity["comment"] = comment
    activity.save()
    activity.new_edge(
        input=activity,
        amount=1.0,
        type=bd.labels.production_edge_default,
    ).save()
    return activity


def lookup_premise(database_name, name, location, product=None):
    lookup = {"database": database_name, "name": name, "location": location}
    if product is not None:
        lookup["product"] = product
    return bd.get_node(**lookup)


for year, premise_database_name in PREMISE_DBS.items():
    case_database_name = CASE_STUDY_BG_DBS[year]
    if case_database_name in bd.databases:
        del bd.databases[case_database_name]
    case_database = bd.Database(case_database_name)
    case_database.register()
    case_database.metadata["representative_time"] = datetime(year, 1, 1).isoformat()

    for code, name, product, location, unit in INTERFACE_SPECS:
        source = premise_node(
            premise_database_name,
            name=name,
            product=product,
            location=location,
            unit=unit,
        )
        interface = create_case_activity(
            case_database,
            code=f"interface_{code}",
            name=name,
            product=product,
            location=location,
            unit=unit,
            comment=f"Case-study interface to {source.key} for support year {year}.",
        )
        interface.new_edge(
            input=source,
            amount=1.0,
            type=bd.labels.consumption_edge_default,
        ).save()

    feedstock_mix = create_case_activity(
        case_database,
        code="steam_cracking_feedstock_mix",
        name=FEEDSTOCK_MIX_NAME,
        product=FEEDSTOCK_MIX_NAME,
        location="RER w/o RU",
        unit="kilogram",
        comment=(
            "One kilogram of the ecoinvent steam-cracker feedstock slate. "
            "Diesel proxies atmospheric gas oil; the seven mass shares sum to one."
        ),
    )
    for name, product, location, unit, source_amount in STEAM_FEEDSTOCK_INPUTS:
        feedstock_mix.new_edge(
            input=premise_node(
                premise_database_name,
                name=name,
                product=product,
                location=location,
                unit=unit,
            ),
            amount=source_amount / STEAM_FEEDSTOCK_TOTAL,
            type=bd.labels.consumption_edge_default,
        ).save()

    chemical_factory_organics_source = premise_node(
        premise_database_name,
        name="chemical factory construction, organics",
        product="chemical factory, organics",
        location="RER",
        unit="unit",
    )
    for code, name, location in [
        ("steam_cracker_installation", STEAM_INSTALLATION_NAME, "RER w/o RU"),
        ("mto_installation", MTO_INSTALLATION_NAME, "RER"),
    ]:
        installation = create_case_activity(
            case_database,
            code=code,
            name=name,
            product=name,
            location=location,
            unit="unit",
            comment=(
                "Route-specific cost identity wrapping one unit of "
                "chemical factory construction, organics."
            ),
        )
        installation.new_edge(
            input=chemical_factory_organics_source,
            amount=1.0,
            type=bd.labels.consumption_edge_default,
        ).save()

    eco2r_installation = create_case_activity(
        case_database,
        code="eco2r_system_installation",
        name=ECO2R_INSTALLATION_NAME,
        product=ECO2R_INSTALLATION_NAME,
        location="DE",
        unit="kilogram",
        comment=(
            "Aggregated eCO2R reactor and separation installation on the "
            "chemical-factory-mass proxy basis."
        ),
    )
    eco2r_installation.new_edge(
        input=premise_node(
            premise_database_name,
            name="chemical factory construction",
            product="chemical factory",
            location="RER",
            unit="kilogram",
        ),
        amount=1.0,
        type=bd.labels.consumption_edge_default,
    ).save()
    eco2r_installation.new_edge(
        input=premise_node(
            premise_database_name,
            name="market for copper, cathode",
            product="copper, cathode",
            location="GLO",
            unit="kilogram",
        ),
        amount=7.4e-11 / 7.32e-7,
        type=bd.labels.consumption_edge_default,
    ).save()
    eco2r_installation.new_edge(
        input=premise_node(
            premise_database_name,
            name="market for steel, low-alloyed",
            product="steel, low-alloyed",
            location="GLO",
            unit="kilogram",
        ),
        amount=3.017679e-6 / 7.32e-7,
        type=bd.labels.consumption_edge_default,
    ).save()

    pei = create_case_activity(
        case_database,
        code="ethylene_case_study_polyethyleneimine",
        name=PEI_NAME,
        product=PEI_PRODUCT,
        location="DE",
        unit="kilogram",
        comment=(
            "Case-study reproduction of the disco2very polyethyleneimine "
            "inventory based on Deutz and Bardow (2021), "
            "https://doi.org/10.1038/s41560-020-00771-9."
        ),
    )
    pei_inputs = [
        ("market for monoethanolamine", "GLO", None, 1.42),
        ("sodium sulfate production, from natural sources", "RER", None, -0.86),
        ("market for sodium hydroxide, without water, in 50% solution state", "RER", None, 3.72),
        ("market for hydrochloric acid, without water, in 30% solution state", "RER", None, 0.170666667),
        ("market for ethanol, without water, in 99.7% solution state, from ethylene", "RER", None, 2.84),
        ("ethanol production, ethylene hydration", "RER", "diethyl ether, without water, in 99.95% solution state", 33.94),
        ("water production, deionised", "Europe without Switzerland", None, 11.99),
        ("market for electricity, medium voltage", "DE", None, 0.42),
        ("heat production, natural gas, at industrial furnace low-NOx >100kW", "Europe without Switzerland", None, 2.35),
        ("market for spent solvent mixture", "Europe without Switzerland", None, -0.37),
    ]
    for name, location, product, amount in pei_inputs:
        pei.new_edge(
            input=lookup_premise(
                premise_database_name,
                name=name,
                location=location,
                product=product,
            ),
            amount=amount,
            type=bd.labels.consumption_edge_default,
        ).save()

    adsorbent = create_case_activity(
        case_database,
        code="ethylene_case_study_adsorbent_amine_on_alumina",
        name=ADSORBENT_NAME,
        product=ADSORBENT_PRODUCT,
        location="DE",
        unit="kilogram",
        comment=(
            "Case-study reproduction of the disco2very amine-on-alumina "
            "adsorbent based on Deutz and Bardow (2021) and Leonzio et al. "
            "(2022), https://doi.org/10.1016/j.spc.2022.04.004."
        ),
    )
    adsorbent.new_edge(
        input=pei,
        amount=0.557735849,
        type=bd.labels.consumption_edge_default,
    ).save()
    adsorbent.new_edge(
        input=lookup_premise(
            premise_database_name,
            name="market for aluminium oxide, metallurgical",
            location="IAI Area, Western and Central Europe",
        ),
        amount=0.442264151,
        type=bd.labels.consumption_edge_default,
    ).save()

    dac_system_4kt = create_case_activity(
        case_database,
        code="ethylene_case_study_dac_solid_sorbent_4kt_construction",
        name=DAC_4KT_NAME,
        product=DAC_4KT_PRODUCT,
        location="RER",
        unit="unit",
        comment=(
            "Construction of one solid-sorbent TVSA direct-air-capture system "
            "with 4 kt CO2/a nominal capacity, reproduced from disco2very on "
            "the basis of Deutz and Bardow (2021). Plant end-of-life is excluded."
        ),
    )
    dac_materials = [
        ("market for concrete, 30MPa", "CH", None, 1501),
        ("market for reinforcing steel", "GLO", None, 217590),
        ("market for stone wool", "GLO", None, 2600),
        ("market for steel, chromium steel 18/8", "GLO", None, 47145),
        ("market for polyurethane, rigid foam", "RER", None, 2600),
        ("market for copper, anode", "GLO", None, 2200),
        ("market for aluminium, primary, ingot", "IAI Area, Western and Central Europe", None, 16000),
        ("market for alkyd paint, white, without solvent, in 60% solution state", "RER", None, 1600),
        ("market for synthetic rubber", "GLO", None, 6300),
        ("sheet rolling, aluminium", "RER", None, 16000),
        ("metal working, average for copper product manufacturing", "RER", "metal working, average for copper product manufacturing", 1600),
        ("sheet rolling, copper", "RER", None, 600),
    ]
    for name, location, product, amount in dac_materials:
        dac_system_4kt.new_edge(
            input=lookup_premise(
                premise_database_name,
                name=name,
                location=location,
                product=product,
            ),
            amount=amount,
            type=bd.labels.consumption_edge_default,
        ).save()
    dac_system_4kt.new_edge(
        input=bd.Database(BIOSPHERE_DB).get(
            code="fe9c3a98-a6d2-452d-a9a4-a13e64f1b95b"
        ),
        amount=1045,
        type=bd.labels.biosphere_edge_default,
    ).save()

    case_database.process()

The names, products, locations and units of the 2020 interface nodes provide
the identities used by optimex to resolve the corresponding activities in later
support years. The original 2020 premise steam-cracking activity is read only to
retain its documented direct biosphere exchanges and verify its input amounts.

In [ ]:
REFERENCE_DB = CASE_STUDY_BG_DBS[2020]
SOURCE_STEAM_DB = PREMISE_DBS[2020]


def case_node(*, name, product, location, unit):
    return bd.get_node(
        database=REFERENCE_DB,
        name=name,
        product=product,
        location=location,
        unit=unit,
    )


steam_cracking_inventory = bd.get_node(
    database=SOURCE_STEAM_DB,
    name="unsaturated hydrocarbons production, steam cracking operation, average",
    product="ethylene",
    location="RER w/o RU",
    unit="kilogram",
)
electricity_mv = case_node(
    name="market for electricity, medium voltage",
    product="electricity, medium voltage",
    location="DE",
    unit="kilowatt hour",
)
heat_pump = case_node(
    name="heat production, at heat pump 30kW, allocation exergy",
    product="heat, central or small-scale, other than natural gas",
    location="Europe without Switzerland",
    unit="megajoule",
)
water_deionized = case_node(
    name="market for water, deionised",
    product="water, deionised",
    location="Europe without Switzerland",
    unit="kilogram",
)
wastewater = case_node(
    name="market for wastewater, unpolluted",
    product="wastewater, unpolluted",
    location="RoW",
    unit="cubic meter",
)
cooling = case_node(
    name="market for cooling energy",
    product="cooling energy",
    location="GLO",
    unit="megajoule",
)
cooling_minus_15 = case_node(
    name="market for cooling energy, at -15 °C",
    product="cooling energy, at -15 °C",
    location="GLO",
    unit="megajoule",
)
cooling_minus_25 = case_node(
    name="cooling energy production, at -25 °C, propylene compression refrigeration system 1 MW",
    product="cooling energy, at -25 °C",
    location="GLO",
    unit="megajoule",
)
cooling_minus_45 = case_node(
    name="market for cooling energy, at -45 °C",
    product="cooling energy, at -45 °C",
    location="GLO",
    unit="megajoule",
)
cooling_minus_55 = case_node(
    name="market for cooling energy, at -55 °C",
    product="cooling energy, at -55 °C",
    location="GLO",
    unit="megajoule",
)
cooling_minus_100 = case_node(
    name="market for cooling energy, at -100 °C",
    product="cooling energy, at -100 °C",
    location="GLO",
    unit="megajoule",
)
industrial_heat = case_node(
    name="market for heat, district or industrial, natural gas",
    product="heat, district or industrial, natural gas",
    location="Europe without Switzerland",
    unit="megajoule",
)
adsorbent_amine_alumina = case_node(
    name=ADSORBENT_NAME,
    product=ADSORBENT_PRODUCT,
    location="DE",
    unit="kilogram",
)
spent_adsorbent_treatment = case_node(
    name="treatment of spent anion exchange resin from potable water production, municipal incineration",
    product="spent anion exchange resin from potable water production",
    location="GLO",
    unit="kilogram",
)
steam_feedstock_mix = case_node(
    name=FEEDSTOCK_MIX_NAME,
    product=FEEDSTOCK_MIX_NAME,
    location="RER w/o RU",
    unit="kilogram",
)
steam_installation = case_node(
    name=STEAM_INSTALLATION_NAME,
    product=STEAM_INSTALLATION_NAME,
    location="RER w/o RU",
    unit="unit",
)
mto_installation = case_node(
    name=MTO_INSTALLATION_NAME,
    product=MTO_INSTALLATION_NAME,
    location="RER",
    unit="unit",
)
eco2r_installation = case_node(
    name=ECO2R_INSTALLATION_NAME,
    product=ECO2R_INSTALLATION_NAME,
    location="DE",
    unit="kilogram",
)
dac_system_4kt = case_node(
    name=DAC_4KT_NAME,
    product=DAC_4KT_PRODUCT,
    location="RER",
    unit="unit",
)
pem_stack = case_node(
    name="electrolyzer production, 1MWe, PEM, Stack",
    product="electrolyzer, 1MWe, PEM, Stack",
    location="RER",
    unit="unit",
)
pem_bop = case_node(
    name="electrolyzer production, 1MWe, PEM, Balance of Plant",
    product="electrolyzer, 1MWe, PEM, Balance of Plant",
    location="RER",
    unit="unit",
)
methanol_facility = case_node(
    name="methanol production facility, construction",
    product="methanol production facility, construction",
    location="RER",
    unit="unit",
)
co2_fossil = bd.get_node(
    database=BIOSPHERE_DB,
    name="Carbon dioxide, fossil",
    categories=("air",),
)

## 4. Products and processes

Four foreground products connect six decision processes. Operation exchanges
scale with operation, while installation exchanges scale with new installations.
All source inventory amounts are retained; named background wrappers only make
the direct operating and installation cost identities unambiguous.

In [ ]:
if FOREGROUND_DB in bd.databases:
    del bd.databases[FOREGROUND_DB]
foreground = bd.Database(FOREGROUND_DB)
foreground.register()

ethylene = foreground.new_node(
    code="ethylene",
    name="Ethylene",
    unit="kilogram",
    type=bd.labels.product_node_default,
)
ethylene.save()

captured_co2 = foreground.new_node(
    code="captured_co2",
    name="Captured carbon dioxide",
    unit="kilogram",
    type=bd.labels.product_node_default,
)
captured_co2.save()

hydrogen = foreground.new_node(
    code="hydrogen",
    name="Hydrogen",
    unit="kilogram",
    type=bd.labels.product_node_default,
)
hydrogen.save()

methanol = foreground.new_node(
    code="methanol",
    name="Methanol",
    unit="kilogram",
    type=bd.labels.product_node_default,
)
methanol.save()

### Steam cracking

The conventional route exposes the direct operating inputs and direct biosphere
exchanges of the ecoinvent steam-cracking dataset. Its seven documented feedstocks
are grouped into one mass-balanced background mix; utilities, auxiliaries and
treatments remain separate. Direct biosphere exchanges are retained unchanged,
while cumulative supply-chain emissions are still calculated through premise.

The route-specific installation wraps one organic chemical factory and retains
the original ecoinvent coefficient at the foreground edge.

In [ ]:
steam = foreground.new_node(
    code="steam_cracking",
    name=PROCESS_NAMES["steam_cracking"],
    location="RER w/o RU",
    type=bd.labels.process_node_default,
    operation_time_limits=(0, LIFETIMES_YEARS["steam_cracking"] - 1),
)
steam.save()

steam_cracking_operating_amounts = [
    ("market for butane", 0.19711504876613617),
    ("market for compressed air, 700 kPa gauge", 0.014663908630609512),
    ("market for diesel", 0.04867038503289223),
    ("market for ethane", 0.05475417897105217),
    ("market for hazardous waste, for incineration", -0.003802229417487979),
    ("market for hazardous waste, for underground deposit", -0.0010579951340332627),
    ("market for inert waste, for final disposal", -0.00037626258563250303),
    ("market for methanol", 0.00023348795366473496),
    ("market for naphtha", 0.7787261009216309),
    ("market for natural gas liquids", 0.02676870860159397),
    ("market for nitrogen, liquid", 0.011272253468632698),
    ("market for propane", 0.09247373044490814),
    ("market for refinery gas", 0.018251389265060425),
    ("market for sodium hydroxide, without water, in 50% solution state", 0.004936636425554752),
    ("market for wastewater, average", -0.0006797355017624795),
    ("market for water, deionised", 2.186876058578491),
    ("market group for electricity, medium voltage", 0.1206742525100708),
]
assert len(steam_cracking_operating_amounts) == 17

feedstock_names = {item[0] for item in STEAM_FEEDSTOCK_INPUTS}
source_technosphere = list(steam_cracking_inventory.technosphere())
source_operating_exchanges = {}
for activity_name, amount in steam_cracking_operating_amounts:
    matches = [
        exchange for exchange in source_technosphere
        if exchange.input.get("name") == activity_name
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Expected one steam-cracking input named '{activity_name}', "
            f"found {len(matches)}."
        )
    source_exchange = matches[0]
    if not np.isclose(float(source_exchange["amount"]), amount):
        raise ValueError(
            f"Unexpected amount for '{activity_name}': "
            f"{source_exchange['amount']} instead of {amount}."
        )
    source_operating_exchanges[activity_name] = source_exchange

assert np.isclose(
    sum(amount for name, amount in steam_cracking_operating_amounts if name in feedstock_names),
    STEAM_FEEDSTOCK_TOTAL,
)

steam_cracking_input_table = pd.DataFrame(
    [
        {
            "source input": name,
            "direct cost flow": (
                FEEDSTOCK_MIX_NAME if name in feedstock_names else name
            ),
            "amount per kg ethylene": amount,
        }
        for name, amount in steam_cracking_operating_amounts
    ]
)

steam_cracking_operating_inputs = [
    {"input": steam_feedstock_mix, "amount": STEAM_FEEDSTOCK_TOTAL}
]
for activity_name, amount in steam_cracking_operating_amounts:
    if activity_name in feedstock_names:
        continue
    source_input = source_operating_exchanges[activity_name].input
    steam_cracking_operating_inputs.append(
        {
            "input": case_node(
                name=source_input.get("name"),
                product=(
                    source_input.get("reference product")
                    or source_input.get("product")
                ),
                location=source_input.get("location"),
                unit=source_input.get("unit"),
            ),
            "amount": amount,
        }
    )
assert len(steam_cracking_operating_inputs) == 11

steam_cracking_biosphere_exchanges = list(steam_cracking_inventory.biosphere())
assert len(steam_cracking_biosphere_exchanges) == 44
steam_cracking_biosphere_table = pd.DataFrame(
    [
        {
            "name": exchange.input.get("name"),
            "categories": exchange.input.get("categories"),
            "unit": exchange.input.get("unit"),
            "amount per kg ethylene": float(exchange["amount"]),
        }
        for exchange in steam_cracking_biosphere_exchanges
    ]
)

display(steam_cracking_input_table)
display(steam_cracking_biosphere_table)

steam.new_edge(
    input=ethylene,
    amount=1.0,
    type=bd.labels.production_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(steam),
).save()
for item in steam_cracking_operating_inputs:
    steam.new_edge(
        input=item["input"],
        amount=item["amount"],
        type=bd.labels.consumption_edge_default,
        operation=True,
        temporal_distribution=infer_operation_td_from_limits(steam),
    ).save()

for exchange in steam_cracking_biosphere_exchanges:
    steam.new_edge(
        input=exchange.input,
        amount=float(exchange["amount"]),
        type=bd.labels.biosphere_edge_default,
        operation=True,
        temporal_distribution=infer_operation_td_from_limits(steam),
    ).save()
steam.new_edge(
    input=steam_installation,
    amount=INSTALLATION_AMOUNTS["steam_factory"],
    type=bd.labels.consumption_edge_default,
    operation=False,
    temporal_distribution=infer_construction_td_from_limits(steam),
).save()

### Direct air capture

DAC produces the shared captured-CO2 product. Electricity, heat, the original
disco2very amine-on-alumina adsorbent, spent-adsorbent treatment and atmospheric
CO2 uptake are operating flows. Construction uses the reproduced 4 kt CO2/a
solid-sorbent system from Deutz and Bardow without its end-of-life exchange.
The source inventory's 20-year lifetime is also used as the Optimex model lifetime.


In [ ]:
dac = foreground.new_node(
    code="dac",
    name=PROCESS_NAMES["dac"],
    location="RER",
    type=bd.labels.process_node_default,
    operation_time_limits=(0, LIFETIMES_YEARS["dac"] - 1),
)
dac.save()

dac.new_edge(
    input=captured_co2,
    amount=1.0,
    type=bd.labels.production_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(dac),
).save()
dac.new_edge(
    input=electricity_mv,
    amount=0.7,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(dac),
).save()
dac.new_edge(
    input=heat_pump,
    amount=4.7,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(dac),
).save()
dac.new_edge(
    input=adsorbent_amine_alumina,
    amount=0.0075,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(dac),
).save()
dac.new_edge(
    input=spent_adsorbent_treatment,
    amount=-0.0075,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(dac),
).save()
dac.new_edge(
    input=co2_fossil,
    amount=-1.0,
    type=bd.labels.biosphere_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(dac),
).save()
dac.new_edge(
    input=dac_system_4kt,
    amount=INSTALLATION_AMOUNTS["dac_system"],
    type=bd.labels.consumption_edge_default,
    operation=False,
    temporal_distribution=infer_construction_td_from_limits(dac),
).save()


### PEM electrolysis

PEM electrolysis produces hydrogen from electricity and deionized water. Stack
and balance-of-plant exchange amounts are copied unchanged from the
`methanol_and_iron` inventory. Their source lifetimes are not needed for
installation scaling. The process uses Tiggeloven's 25-year AEC plant
lifetime as a proxy; the separate 9-year stack replacement is not modeled.


In [ ]:
pem = foreground.new_node(
    code="pem",
    name=PROCESS_NAMES["pem"],
    location="RER",
    type=bd.labels.process_node_default,
    operation_time_limits=(0, LIFETIMES_YEARS["pem"] - 1),
)
pem.save()

pem.new_edge(
    input=hydrogen,
    amount=1.0,
    type=bd.labels.production_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(pem),
).save()
pem.new_edge(
    input=electricity_mv,
    amount=56.00509259,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(pem),
).save()
pem.new_edge(
    input=water_deionized,
    amount=8.936011905,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(pem),
).save()
pem.new_edge(
    input=wastewater,
    amount=-2.2e-5,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(pem),
).save()
pem.new_edge(
    input=pem_stack,
    amount=INSTALLATION_AMOUNTS["pem_stack"],
    type=bd.labels.consumption_edge_default,
    operation=False,
    temporal_distribution=infer_construction_td_from_limits(pem),
).save()
pem.new_edge(
    input=pem_bop,
    amount=INSTALLATION_AMOUNTS["pem_bop"],
    type=bd.labels.consumption_edge_default,
    operation=False,
    temporal_distribution=infer_construction_td_from_limits(pem),
).save()


### CO2 hydrogenation

CO2 hydrogenation combines the shared captured-CO2 and hydrogen products to
produce methanol. Electricity and wastewater are operating flows; the methanol
facility is a separate installation with its unchanged inventory amount.


In [ ]:
co2_hydrogenation = foreground.new_node(
    code="co2_hydrogenation",
    name=PROCESS_NAMES["co2_hydrogenation"],
    location="RER",
    type=bd.labels.process_node_default,
    operation_time_limits=(0, LIFETIMES_YEARS["co2_hydrogenation"] - 1),
)
co2_hydrogenation.save()

co2_hydrogenation.new_edge(
    input=methanol,
    amount=1.0,
    type=bd.labels.production_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(co2_hydrogenation),
).save()
co2_hydrogenation.new_edge(
    input=electricity_mv,
    amount=1.32878456384964,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(co2_hydrogenation),
).save()
co2_hydrogenation.new_edge(
    input=captured_co2,
    amount=1.435820454,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(co2_hydrogenation),
).save()
co2_hydrogenation.new_edge(
    input=hydrogen,
    amount=0.197319687,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(co2_hydrogenation),
).save()
co2_hydrogenation.new_edge(
    input=wastewater,
    amount=-0.000562265917602996,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(co2_hydrogenation),
).save()
co2_hydrogenation.new_edge(
    input=methanol_facility,
    amount=INSTALLATION_AMOUNTS["methanol_factory"],
    type=bd.labels.consumption_edge_default,
    operation=False,
    temporal_distribution=infer_construction_td_from_limits(co2_hydrogenation),
).save()


### Methanol-to-olefins

MTO converts methanol to ethylene using the mass-allocated disco2very inventory
values. The -30 °C and -75 °C cooling duties use the available -25 °C and
-100 °C premise cooling processes as visible operating proxies. The organic
chemical factory is installed separately.


In [ ]:
mto = foreground.new_node(
    code="mto",
    name=PROCESS_NAMES["mto"],
    location="RER",
    type=bd.labels.process_node_default,
    operation_time_limits=(0, LIFETIMES_YEARS["mto"] - 1),
)
mto.save()

mto.new_edge(
    input=ethylene,
    amount=1.0,
    type=bd.labels.production_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(mto),
).save()
mto.new_edge(
    input=electricity_mv,
    amount=0.1428058844,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(mto),
).save()
mto.new_edge(
    input=cooling,
    amount=0.17224,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(mto),
).save()
mto.new_edge(
    input=cooling_minus_25,
    amount=0.4304 + 0.9216,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(mto),
).save()
mto.new_edge(
    input=cooling_minus_100,
    amount=0.21528,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(mto),
).save()
mto.new_edge(
    input=wastewater,
    amount=-0.0012778376,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(mto),
).save()
mto.new_edge(
    input=methanol,
    amount=2.3920583664,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(mto),
).save()
mto.new_edge(
    input=mto_installation,
    amount=INSTALLATION_AMOUNTS["mto_factory"],
    type=bd.labels.consumption_edge_default,
    operation=False,
    temporal_distribution=infer_construction_td_from_limits(mto),
).save()

### Aggregated eCO2R ethylene production

Electrochemical CO2 reduction and product separation are represented by one
decision process because the available CAPEX proxy covers the combined system.
The operating amounts remain visibly traceable to the two source inventories.
The `6.091081 kg CO2/kg ethylene` emission represents oxidation of process
by-products, not ethylene product end-of-life.

The installation wrapper combines the source factory, copper and separator-steel
amounts on the chemical-factory-mass basis without changing their net quantities.

In [ ]:
eco2r = foreground.new_node(
    code="eco2r",
    name=PROCESS_NAMES["eco2r"],
    location="DE",
    type=bd.labels.process_node_default,
    operation_time_limits=(0, LIFETIMES_YEARS["eco2r"] - 1),
)
eco2r.save()

eco2r.new_edge(
    input=ethylene,
    amount=1.0,
    type=bd.labels.production_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(eco2r),
).save()
eco2r.new_edge(
    input=electricity_mv,
    amount=74.878 + (0.133897031 + 0.5324327),
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(eco2r),
).save()
eco2r.new_edge(
    input=water_deionized,
    amount=4.4249,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(eco2r),
).save()
eco2r.new_edge(
    input=captured_co2,
    amount=9.229,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(eco2r),
).save()
eco2r.new_edge(
    input=cooling,
    amount=0.010118339 + 3.06967977 + 2.0085,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(eco2r),
).save()
eco2r.new_edge(
    input=industrial_heat,
    amount=28.487337 + 3.079241549,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(eco2r),
).save()
for cooling_input, amount in [
    (cooling_minus_15, 0.17912),
    (cooling_minus_25, 0.057779),
    (cooling_minus_45, 0.11556),
    (cooling_minus_55, 0.057779),
    (cooling_minus_100, 0.60715),
]:
    eco2r.new_edge(
        input=cooling_input,
        amount=amount,
        type=bd.labels.consumption_edge_default,
        operation=True,
        temporal_distribution=infer_operation_td_from_limits(eco2r),
    ).save()
eco2r.new_edge(
    input=wastewater,
    amount=-0.000763598,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(eco2r),
).save()
eco2r.new_edge(
    input=co2_fossil,
    amount=2.361481 + 3.7296,
    type=bd.labels.biosphere_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(eco2r),
).save()
eco2r.new_edge(
    input=eco2r_installation,
    amount=INSTALLATION_AMOUNTS["eco2r_factory"],
    type=bd.labels.consumption_edge_default,
    operation=False,
    temporal_distribution=infer_construction_td_from_limits(eco2r),
).save()

processes = {
    "steam_cracking": steam,
    "dac": dac,
    "pem": pem,
    "co2_hydrogenation": co2_hydrogenation,
    "mto": mto,
    "eco2r": eco2r,
}

## 5. Demand and existing capacity

The system must supply `1 Mt` of ethylene per year. Two equal steam-cracking
vintages provide the full initial annual capacity; their installation years are
visible proxies to be reviewed with the final lifetime assumptions.


In [ ]:
SYSTEM_YEARS = np.arange(2025, 2051)
ANNUAL_ETHYLENE_DEMAND_KG = 1e9
ethylene_demand = TemporalDistribution(
    date=np.array(
        [datetime(int(year), 1, 1).isoformat() for year in SYSTEM_YEARS],
        dtype="datetime64[s]",
    ),
    amount=np.full(len(SYSTEM_YEARS), ANNUAL_ETHYLENE_DEMAND_KG),
)
functional_demand = {ethylene: ethylene_demand}

# PR #61 semantics: existing_capacity stores process units. One steam-cracker
# unit delivers the 1 kg production exchange over its configured lifetime.
steam_cracking_lifetime_years = int(LIFETIMES_YEARS["steam_cracking"])
steam_cracking_annual_output_per_unit_kg = 1.0 / steam_cracking_lifetime_years
brownfield_annual_capacity_per_vintage_kg = float(
    assumptions.loc["brownfield.capacity_per_vintage", "value"]
)
brownfield_units_per_vintage = (
    brownfield_annual_capacity_per_vintage_kg
    / steam_cracking_annual_output_per_unit_kg
)

existing_capacities = {
    (
        "steam_cracking",
        int(assumptions.loc["brownfield.vintage_1", "value"]),
    ): brownfield_units_per_vintage,
    (
        "steam_cracking",
        int(assumptions.loc["brownfield.vintage_2", "value"]),
    ): brownfield_units_per_vintage,
}

brownfield_table = pd.DataFrame(
    [
        {
            "process": PROCESS_NAMES[process],
            "installation_year": year,
            "installed_process_units": units,
            "annual_capacity_kg_per_year": (
                units * steam_cracking_annual_output_per_unit_kg
            ),
        }
        for (process, year), units in existing_capacities.items()
    ]
)
assert np.allclose(
    brownfield_table["annual_capacity_kg_per_year"],
    brownfield_annual_capacity_per_vintage_kg,
)
assert np.isclose(
    brownfield_table["annual_capacity_kg_per_year"].sum(),
    2 * brownfield_annual_capacity_per_vintage_kg,
)
brownfield_table


## 6. Market-price inputs

The versioned CSV contains price trajectories and source metadata for direct
operating and installation purchases. Prices are attached only to the four
case-study background databases. Their internal premise inputs remain unpriced
by optimex, which prevents double counting.

Missing prices retain the existing warning-based, user-responsibility handling.

In [ ]:
from optimex.economics import set_market_prices

flow_metadata = {}
flow_classes = {}
for process_code, process_node in processes.items():
    for exchange in process_node.technosphere():
        if exchange.input.get("database") == FOREGROUND_DB:
            continue
        product_name = (
            exchange.input.get("reference product")
            or exchange.input.get("product")
        )
        identity = (
            exchange.input.get("name"),
            product_name,
            exchange.input.get("location"),
            exchange.input.get("unit"),
        )
        flow_metadata[identity] = {
            "name": identity[0],
            "product": identity[1],
            "location": identity[2],
            "unit": identity[3],
        }
        flow_classes.setdefault(identity, set()).add(
            "op" if exchange.get("operation") else "cap"
        )

required_cost_flows = pd.DataFrame(
    [
        {
            **flow_metadata[identity],
            "cost_class": (
                "cap_and_op"
                if classes == {"cap", "op"}
                else next(iter(classes))
            ),
        }
        for identity, classes in flow_classes.items()
    ]
).sort_values(["cost_class", "name"], ignore_index=True)

repo_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
cost_csv = (
    repo_root / "notebooks" / "data" / "ethylene_case_study" / "cost_inputs.csv"
)
if not cost_csv.exists():
    raise FileNotFoundError(f"Missing versioned cost input file: {cost_csv}")

cost_inputs = pd.read_csv(cost_csv)
identity_columns = ["name", "product", "location", "unit"]
csv_coverage = cost_inputs[
    identity_columns + ["status"]
].drop_duplicates(identity_columns)
cost_coverage = required_cost_flows.merge(
    csv_coverage,
    on=identity_columns,
    how="left",
)
missing_prices = cost_coverage[cost_coverage["status"].isna()]
if not missing_prices.empty:
    print("WARNING: Missing market-price rows for these foreground flows:")
    display(missing_prices)

set_market_prices(
    price_data=cost_inputs,
    background_databases=CASE_STUDY_BG_DBS,
    product_col="product",
    unit_col="unit",
    strict=False,
)

display(cost_coverage)
cost_inputs.groupby("status").size().rename("CSV rows").to_frame()

## 7. Time-explicit LCA data

The foreground and annual demand are converted into the tensors used by
optimex. The resulting operating and installation cost-flow lists are shown as
the final price-input audit.


In [ ]:
from optimex import converter, lca_processor

METHOD_CLIMATE_CHANGE = (
    "IPCC 2021",
    "climate change",
    "GWP 100a, incl. H and bio CO2",
)
lca_config = lca_processor.LCAConfig(
    demand=functional_demand,
    temporal={
        "start_date": datetime(2025, 1, 1),
        "temporal_resolution": "year",
        "time_horizon": 100,
        "database_dates": {
            database_name: datetime(year, 1, 1)
            for year, database_name in CASE_STUDY_BG_DBS.items()
        },
    },
    characterization_methods=[
        {
            "category_name": "climate_change",
            "brightway_method": METHOD_CLIMATE_CHANGE,
        }
    ],
)
lca_data_processor = lca_processor.LCADataProcessor(
    lca_config,
    foreground_db_name=FOREGROUND_DB,
)


def processor_cost_flow_table(processor):
    rows = []
    all_codes = (
        processor.cost_relevant_cap_flows
        | processor.cost_relevant_op_flows
    )
    for flow_code in sorted(all_codes):
        metadata = processor.intermediate_flows[flow_code]
        rows.append(
            {
                "flow_code": flow_code,
                **metadata,
                "cost_class": (
                    "cap_and_op"
                    if flow_code in processor.cost_relevant_cap_flows
                    and flow_code in processor.cost_relevant_op_flows
                    else "cap"
                    if flow_code in processor.cost_relevant_cap_flows
                    else "op"
                ),
            }
        )
    return pd.DataFrame(rows)


cost_relevant_flows = processor_cost_flow_table(lca_data_processor)
csv_identities = cost_inputs[
    ["name", "product", "location", "unit", "status"]
].drop_duplicates()
cost_relevant_flows = cost_relevant_flows.merge(
    csv_identities,
    on=["name", "product", "location", "unit"],
    how="left",
)

print(
    "cost_relevant_op_flows:",
    len(lca_data_processor.cost_relevant_op_flows),
)
print(
    "cost_relevant_cap_flows:",
    len(lca_data_processor.cost_relevant_cap_flows),
)
display(
    cost_relevant_flows[
        ["cost_class", "name", "product", "location", "unit", "status"]
    ].sort_values(["cost_class", "name"], ignore_index=True)
)
cost_relevant_flows.groupby("status", dropna=False).size().rename(
    "cost-relevant flows"
).to_frame()

In [ ]:
manager = converter.ModelInputManager()
optimization_inputs = manager.parse_from_lca_processor(lca_data_processor)
scenario_inputs = manager.override(
    existing_capacity=existing_capacities,
    vintage_improvements=None,
    discount_rate=float(
        assumptions.loc["economics.discount_rate", "value"]
    ),
    discount_reference_year=2025,
)

In [ ]:
manager.save("data/2026-07-18_model_inputs_ethylene.json")

## 8. Optimization scenarios

The first scenario minimizes cumulative climate-change impact. The second
minimizes discounted total cost without a CO2 price. Both use the same demand,
technology assumptions and existing steam-cracking capacity.


In [ ]:
from optimex import optimizer

SOLVER = "highs"


def solve_scenario(name, objective):
    model = optimizer.create_model(
        scenario_inputs.model_copy(deep=True),
        name=name,
        objective_category="climate_change",
        objective=objective,
    )
    return optimizer.solve_model(
        model,
        solver_name=SOLVER,
        tee=False,
    )


climate_model, climate_objective, climate_results = solve_scenario(
    "ethylene_climate_minimum",
    "environmental",
)
cost_model, cost_objective, cost_results = solve_scenario(
    "ethylene_cost_minimum_without_co2_price",
    "cost",
)


## 9. Results

The summary reports both objective values. The plots compare ethylene
production by route and newly installed annual capacity for the two scenarios.
Cost results are provisional while proxy or placeholder inputs remain.


In [ ]:
from optimex import postprocessing


def cumulative_climate_impact(model):
    return (
        pyo.value(model.total_impact["climate_change"])
        * model.scales["foreground"]
        * model.scales["characterization"]["climate_change"]
    )


result_summary = pd.DataFrame(
    [
        {
            "scenario": "Climate minimum",
            "cumulative_climate_kg_CO2eq": climate_objective,
            "discounted_cost_EUR_2025": pyo.value(climate_model.total_cost),
        },
        {
            "scenario": "Cost minimum without CO2 price",
            "cumulative_climate_kg_CO2eq": cumulative_climate_impact(cost_model),
            "discounted_cost_EUR_2025": cost_objective,
        },
    ]
).set_index("scenario")
result_summary


In [ ]:
plot_colors = {
    "steam_cracking": "#555555",
    "dac": "#3B82A0",
    "pem": "#2D9C7A",
    "co2_hydrogenation": "#73A942",
    "mto": "#B7A12A",
    "eco2r": "#4C78A8",
}


def scenario_tables(model):
    processor = postprocessing.PostProcessor(model)
    production = processor.get_production()
    ethylene_columns = [
        column for column in production.columns if column[1] == "ethylene"
    ]
    route_production = production[ethylene_columns].copy() / 1e9
    route_production.columns = [column[0] for column in ethylene_columns]
    route_production = route_production.loc[
        :, (route_production.abs() > 1e-9).any(axis=0)
    ]

    installation = processor.get_installation().copy() / 1e9
    installation = installation.loc[
        :, (installation.abs() > 1e-9).any(axis=0)
    ]
    return route_production, installation


scenario_models = {
    "Climate minimum": climate_model,
    "Cost minimum without CO2 price": cost_model,
}
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex="col")

for column_index, (title, model) in enumerate(scenario_models.items()):
    production, installation = scenario_tables(model)
    production.rename(columns=PROCESS_NAMES).plot.area(
        ax=axes[0, column_index],
        stacked=True,
        color=[plot_colors[key] for key in production.columns],
        linewidth=0,
    )
    axes[0, column_index].set_title(title)
    axes[0, column_index].set_ylabel("Ethylene production [Mt/a]")
    axes[0, column_index].set_ylim(bottom=0)

    installation.rename(columns=PROCESS_NAMES).plot.bar(
        ax=axes[1, column_index],
        stacked=True,
        color=[plot_colors[key] for key in installation.columns],
        width=0.85,
    )
    axes[1, column_index].set_ylabel("New capacity [Mt reference product/a]")
    axes[1, column_index].set_xlabel("Installation year")
    axes[1, column_index].tick_params(axis="x", rotation=60)

for axis in axes.flat:
    axis.grid(axis="y", alpha=0.25)
    axis.legend(fontsize=8, frameon=False)

plt.tight_layout()
plt.show()

## 10. Interpretation boundary

All assumptions marked `replacement_needed=True` and all `PROXY` or
`PLACEHOLDER` prices must be reviewed before the results are used in the
Bachelor thesis. The route-specific installation prices remain technical
placeholders until the researched TPC values are converted to their wrapper
functional units. Installation-edge amounts remain unchanged from their source
inventories; model lifetimes are separate Optimex assumptions.

CO2 pricing, emissions budgets, Pareto analysis and vintage improvements are
later extensions and are not part of this baseline.